# Student Notebook 02 — Calibration (Layer 2)

Take the model from notebook 01 and make its probability
outputs honest. A model that says "0.7" should be right
about 70% of the time on the films it rates at 0.7.

**Run order:** notebook 01 must run first (we read its
``student_model.joblib``). After this notebook,
``student_calibrated.joblib`` lands in the same folder for
notebook 03 to use.

**What you'll get out:** ECE before vs after calibration
(lower is better), reliability diagram, the calibrated model
saved.

## CONFIG — edit me

The big knob is the calibration method. ``isotonic`` is more
flexible (a step-wise fit) and usually wins on enough data;
``sigmoid`` (Platt scaling) is more constrained but
regularizes well on small samples.

In [ ]:
CONFIG = {
    # MUST match the run_name you used in 01_modeling.ipynb
    # so this notebook reads YOUR model. Each teammate has
    # their own run_name.
    "run_name": "team_baseline_rf",

    "method": "isotonic",     # "sigmoid" | "isotonic"
    "n_bins_for_ece": 10,     # 10 is the standard
}

## Imports + load notebook 01's artifact

In [ ]:
# --- Paths (works from any working directory) ---
from pathlib import Path

def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "docs" / "PROJECT_CONTEXT.md").is_file():
            return cand
    raise RuntimeError(f"Could not locate project root from {Path.cwd()!s}")

PROJECT_ROOT = _find_project_root()
DATA = PROJECT_ROOT / "data" / "processed"
STUDENT = DATA / "student" / CONFIG["run_name"]
STUDENT.mkdir(parents=True, exist_ok=True)
print(f"Project root:  {PROJECT_ROOT}")
print(f"Run name:      {CONFIG['run_name']!r}")
print(f"Artifacts go:  {STUDENT.relative_to(PROJECT_ROOT)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss, log_loss

bundle = joblib.load(STUDENT / "student_model.joblib")
print("Loaded model from notebook 01:")
print(f"  family       = {bundle['config']['model_family']}")
print(f"  target       = {bundle['config']['target']}")
print(f"  feature_set  = {bundle['config']['feature_set']}")
print(f"  CV AUC       = {bundle['cv_auc_mean']:.3f} ± {bundle['cv_auc_std']:.3f}")
print(f"  cal AUC      = {bundle['cal_auc']:.3f}")

## Rebuild the cal-set features

Same code path as notebook 01, just on the cal split.

In [ ]:
feat = pd.read_parquet(DATA / "features.parquet").reset_index()
df_cal = feat[feat["split"] == "cal"].reset_index(drop=True)
feat_cols = bundle["feature_columns"]
target_col = bundle["config"]["target"]

X_cal = df_cal[feat_cols].fillna(0).values
y_cal = df_cal[target_col].astype(int).values
print(f"Cal set: {len(df_cal)} films, positive rate {y_cal.mean():.3f}")

## Uncalibrated probabilities (before)

These come straight from the model's ``predict_proba``. We
compute the Expected Calibration Error (ECE), which is the
weighted-by-bin-count average of |predicted probability −
empirical positive rate|. Lower is better; 0 is perfect.

In [ ]:
def ece(y_true, y_prob, n_bins=10):
    # Quantile-based bins (equal count per bin) avoid empty bins.
    edges = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = 0.0, 1.0
    edges = np.maximum.accumulate(edges)
    bin_idx = np.clip(np.digitize(y_prob, edges[1:-1]), 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.any():
            total += mask.mean() * abs(y_prob[mask].mean() - y_true[mask].mean())
    return float(total)

proba_uncal = bundle["model"].predict_proba(X_cal)[:, 1]
ece_uncal = ece(y_cal, proba_uncal, n_bins=CONFIG["n_bins_for_ece"])
brier_uncal = brier_score_loss(y_cal, proba_uncal)
print(f"BEFORE calibration: ECE = {ece_uncal:.4f}, Brier = {brier_uncal:.4f}")

## Calibrated probabilities (after)

``CalibratedClassifierCV`` with ``FrozenEstimator`` re-fits
the calibrator on a held-out portion of the cal split, so
the calibrator never sees the same film twice. (sklearn 1.6+
replaces the old ``cv="prefit"`` argument with
``FrozenEstimator``.)

In [ ]:
calibrated = CalibratedClassifierCV(
    FrozenEstimator(bundle["model"]),
    method=CONFIG["method"],
    cv=5,
)
# Fit the calibrator only — the underlying model is frozen.
calibrated.fit(X_cal, y_cal)
proba_cal = calibrated.predict_proba(X_cal)[:, 1]
ece_cal = ece(y_cal, proba_cal, n_bins=CONFIG["n_bins_for_ece"])
brier_cal = brier_score_loss(y_cal, proba_cal)
print(f"AFTER  calibration: ECE = {ece_cal:.4f}, Brier = {brier_cal:.4f}")
print(f"Improvement: ECE {ece_uncal:.4f} → {ece_cal:.4f}  ({(ece_uncal - ece_cal)/ece_uncal*100:+.0f}%)")

## Reliability diagram

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for label, p, color in [("uncalibrated", proba_uncal, "#d7301f"),
                         (CONFIG["method"], proba_cal, "#2c7fb8")]:
    mean_pred, frac_pos = calibration_curve(y_cal, p, n_bins=CONFIG["n_bins_for_ece"],
                                             strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", linewidth=2, color=color, label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect")
ax.set_xlabel("Predicted probability (mean per bin)")
ax.set_ylabel("Empirical positive rate")
ax.set_title(f"Reliability diagram — {bundle['config']['model_family']} on {bundle['config']['target']}")
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()

## Save the calibrated model

In [ ]:
out = {
    **bundle,
    "calibration_method": CONFIG["method"],
    "calibrated_model": calibrated,
    "ece_uncalibrated": ece_uncal,
    "ece_calibrated": ece_cal,
    "brier_uncalibrated": brier_uncal,
    "brier_calibrated": brier_cal,
}
joblib.dump(out, STUDENT / "student_calibrated.joblib")
print(f"Saved {STUDENT / 'student_calibrated.joblib'}")

## Compare to your teammates

| Calibration method | Typical ECE drop |
|---|---|
| sigmoid | small but reliable |
| isotonic | larger; needs enough cal samples |

On the rigorous pipeline (Phase 5) ECE on roi_gt_2 dropped
from 0.243 to 0.108 with isotonic calibration. Your numbers
will be in that ballpark.

Next notebook (03_decision) will use these calibrated
probabilities to recommend Greenlight / Pass / Refer.